# Feature Engineering and Resampling Checks

Task 1 notebook for verifying engineered features, preprocessing transformations, and train-only SMOTE class balance.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CREDITCARD, FRAUD_DATA, IP_COUNTRY
from src.data_utils import basic_clean, load_creditcard, load_fraud_data, load_ip_country
from src.features import add_country_by_ip, engineer_creditcard_features, engineer_fraud_features
from src.preprocessing import apply_smote, build_preprocessor, prepare_modeling_frame, stratified_split


In [ ]:
fraud = engineer_fraud_features(add_country_by_ip(basic_clean(load_fraud_data(FRAUD_DATA)), load_ip_country(IP_COUNTRY)))
credit = engineer_creditcard_features(basic_clean(load_creditcard(CREDITCARD)))
fraud.shape, credit.shape

In [ ]:
fraud_features = ['hour_of_day', 'day_of_week', 'time_since_signup_hours', 'user_transaction_count', 'device_transaction_count', 'purchase_value_log1p', 'country']
fraud[fraud_features + ['class']].head()

In [ ]:
credit_features = ['hour_of_day', 'amount_log1p', 'Amount', 'Class']
credit[credit_features].head()

In [ ]:
def smote_check(df, target):
    modeling_df = prepare_modeling_frame(df, target)
    split = stratified_split(modeling_df, target)
    preprocessor = build_preprocessor(split.X_train)
    X_train = preprocessor.fit_transform(split.X_train)
    _, y_resampled = apply_smote(X_train, split.y_train)
    return pd.DataFrame({
        'before_train_smote': split.y_train.value_counts().sort_index(),
        'after_train_smote': pd.Series(y_resampled).value_counts().sort_index(),
        'heldout_test_unchanged': split.y_test.value_counts().sort_index(),
    })

smote_check(fraud, 'class')

In [ ]:
smote_check(credit, 'Class')

SMOTE is applied after the stratified split and after preprocessing on the training fold only. The held-out test distribution is intentionally left unchanged to preserve a realistic evaluation set.